# SVG Parsing to Intermediate

### Schema:
```pl

  {
    "metadata": {
      "title": str | None,              # Chart title
      "chartType": str,                 # "line", "bar", "scatter", "pie", etc.
      "inferredType": str | None,       # If auto-detected, confidence level
    },

    "axes": {
      "x": {
        "label": str | None,            # "Year", "Country", etc.
        "type": str,                    # "quantitative", "temporal", "nominal", "ordinal"
        "scale": str | None,            # "linear", "log", "time"
        "domain": [min, max] | None,    # Value range
        "ticks": [str | float] | None   # Actual tick labels extracted
      },
      "y": { /* same structure */ }
    },

    "data": {
      "series": [
        {
          "name": str | None,           # Series label (for multi-series charts)
          "encoding": {
            "x": str,                   # Field name mapped to x
            "y": str,                   # Field name mapped to y
            "color": str | None,        # Color encoding if present
            "size": str | None          # Size encoding if present
          },
          "values": [
            {"x": value, "y": value, "label": str | None},
            # ... extracted data points
          ],
          "style": {
            "color": str | None,        # Hex color
            "strokeWidth": float | None,
            "markType": str             # "line", "point", "bar", "area"
          }
        }
      ]
    },

    "legend": {
      "position": str | None,           # "right", "bottom", etc.
      "items": [
        {"label": str, "color": str, "shape": str | None}
      ]
    } | None,

    "annotations": [
      {
        "type": str,                    # "text", "arrow", "callout"
        "text": str,
        "position": {"x": float, "y": float} | None,
        "target": str | None            # What data point it refers to
      }
    ] | None,

    "context": {
      "ariaLabel": str | None,          # From your SvgData
      "ariaDescribedBy": str | None,
      "parentContext": str | None       # Surrounding page text
    }
  }
```

### Design rationale:

  - Semantic-first: Chart type, axis types, field encodings are explicit
  - Data extraction: Actual values separated from visual properties
  - Minimal visual info: Only color/style where it carries semantic meaning (e.g., line color for series distinction)
  - LLM-friendly: Structure mirrors how you'd describe a chart verbally
  - Extensible: Easy to add fields without breaking existing parsers



### Example:   output for that NFL ticket price chart:

```json
{
    "metadata": {
      "title": "National Football League average ticket price from 2006 to 2019 (in U.S. dollars)",
      "chartType": "line",
      "inferredType": "temporal-trend"
    },
    "axes": {
      "x": {
        "label": "Year",
        "type": "temporal",
        "scale": "time",
        "domain": [2006, 2018],
        "ticks": ["2006", "2008", "2010", "2012", "2014", "2016", "2018"]
      },
      "y": {
        "label": "Ticket price in U.S. dollars",
        "type": "quantitative",
        "scale": "linear",
        "domain": [0, 100],
        "ticks": ["0", "20", "40", "60", "80", "100"]
      }
    },
    "data": {
      "series": [{
        "name": null,
        "encoding": {"x": "year", "y": "price"},
        "values": [
          {"x": 2005, "y": 62.38},
          {"x": 2006, "y": 67.11},
          // ...
        ],
        "style": {"color": "#c4c4c4", "markType": "line"}
      }]
    },
    "legend": null,
    "annotations": null
  }

```

# Synthetic Example

In [1]:
svg_html = """
  <svg xmlns="http://www.w3.org/2000/svg" width="400" height="300" viewBox="0 0 400 300">
    <!-- Title -->
    <text x="200" y="30" font-size="16" font-weight="bold" text-anchor="middle">
      Monthly Sales 2024
    </text>

    <!-- Y-axis labels -->
    <text x="45" y="250" font-size="12" text-anchor="end">0</text>
    <text x="45" y="200" font-size="12" text-anchor="end">50</text>
    <text x="45" y="150" font-size="12" text-anchor="end">100</text>
    <text x="45" y="100" font-size="12" text-anchor="end">150</text>
    <text x="45" y="50" font-size="12" text-anchor="end">200</text>

    <!-- X-axis labels -->
    <text x="90" y="270" font-size="12" text-anchor="middle">Jan</text>
    <text x="150" y="270" font-size="12" text-anchor="middle">Feb</text>
    <text x="210" y="270" font-size="12" text-anchor="middle">Mar</text>
    <text x="270" y="270" font-size="12" text-anchor="middle">Apr</text>
    <text x="330" y="270" font-size="12" text-anchor="middle">May</text>

    <!-- Bars -->
    <rect x="70" y="170" width="40" height="80" fill="#4a90e2"/>
    <rect x="130" y="130" width="40" height="120" fill="#4a90e2"/>
    <rect x="190" y="190" width="40" height="60" fill="#4a90e2"/>
    <rect x="250" y="110" width="40" height="140" fill="#4a90e2"/>
    <rect x="310" y="150" width="40" height="100" fill="#4a90e2"/>

    <!-- Axes -->
    <line x1="50" y1="50" x2="50" y2="250" stroke="black" stroke-width="2"/>
    <line x1="50" y1="250" x2="350" y2="250" stroke="black" stroke-width="2"/>
  </svg>
"""

In [ ]:
from parser import VizParser
from schemas import ChartContext

In [ ]:
parser = VizParser(svg_html)

context = ChartContext(
      ariaLabel="chart",
      parentContext="meow meow"
  )

In [ ]:
chart = parser.parse_svg(svg_html, context)

print(chart.model_dump_json(indent=2))

{
  "metadata": {
    "title": "Monthly Sales 2024",
    "chartType": "bar",
    "inferredType": null
  },
  "axes": {
    "y": {
      "label": null,
      "type": "quantitative",
      "scale": "linear",
      "domain": [
        0.0,
        200.0
      ],
      "ticks": [
        "0",
        "50",
        "100",
        "150",
        "200"
      ]
    }
  },
  "data": {
    "series": [
      {
        "name": null,
        "encoding": {
          "x": "category",
          "y": "value"
        },
        "values": [
          {
            "x": 0.0,
            "y": 80.0,
            "label": null
          },
          {
            "x": 1.0,
            "y": 120.0,
            "label": null
          },
          {
            "x": 2.0,
            "y": 60.0,
            "label": null
          },
          {
            "x": 3.0,
            "y": 140.0,
            "label": null
          },
          {
            "x": 4.0,
            "y": 100.0,
            "label": null
  